# 13 · Síntese: a linha do tempo, o que o vetor carrega e o roteiro de 4 h

> **Disciplina:** Representações de Linguagem — *como transformar palavra em vetor, o que esse vetor carrega e o viés que vem junto* (BLIS 2S26).
> Este notebook faz parte de uma sequência (`00` → `13`). Código de apoio em `src/replang`; interface interativa em `app/` (`uv run replang app`).


Este notebook fecha a sequência. Ele (1) consolida em uma tabela única os modelos e as métricas vistas, (2) reconta a história em uma linha do tempo, (3) responde às três perguntas do título da apresentação e (4) traz o roteiro minuto a minuto (detalhado em `docs/roteiro_4h.md`).

In [1]:
# Configuração comum a todos os notebooks ---------------------------------------------
import sys, warnings, os
from pathlib import Path
ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import plotly.io as pio
pio.renderers.default = "plotly_mimetype+notebook_connected"  # JupyterLab renderiza offline; HTML carrega plotly.js via CDN (notebooks leves)
pd.set_option("display.max_colwidth", 80); pd.set_option("display.width", 160)
from replang import PATHS, WordVectors, settings
FAST = settings.fast  # REPLANG_FAST=1 reduz épocas/tamanhos
print("raiz:", ROOT, "| fast:", FAST)

raiz: Y:\BLIS\linguagemBLIS2S26\linguagem-blis2s26 | fast: False


## 1. Linha do tempo

In [2]:
timeline = pd.DataFrame([
    (1954, "Harris; Firth (1957)", "hipótese distribucional", "premissa", "01"),
    (1990, "Deerwester et al. (LSA)", "SVD da matriz termo×documento", "contagem", "01"),
    (1996, "Lund & Burgess (HAL)", "matriz palavra×palavra por janela", "contagem", "01"),
    (2003, "Bengio et al. (NNLM)", "modelo neural de linguagem com embeddings", "preditivo", "02"),
    (2013, "Mikolov et al. (2013a)", "CBOW e Skip-gram; analogias", "preditivo", "02"),
    (2013, "Mikolov et al. (2013b)", "NEG, HS, subamostragem, frases, composição aditiva", "preditivo", "03"),
    (2014, "Pennington et al. (GloVe)", "mínimos quadrados ponderados sobre log X_ij", "contagem+predição", "04"),
    (2014, "Le & Mikolov (Doc2Vec)", "vetor por parágrafo", "preditivo", "11"),
    (2014, "Levy & Goldberg", "SGNS = fatoração de PMI deslocado", "teoria", "01/03"),
    (2016, "Bolukbasi et al.", "viés de gênero como direção; hard/soft debias", "análise/ética", "08/09/10"),
    (2017, "Bojanowski et al. (fastText)", "n-gramas de caracteres; OOV", "subpalavra", "05"),
    (2017, "Hartmann et al.", "31 modelos PT; analogias × tarefas", "avaliação", "06/07"),
    (2018, "Peters et al. (ELMo)", "representações contextuais de biLM", "contextual", "12"),
    (2018, "Devlin et al. (BERT)", "Transformers bidirecionais + fine-tuning", "contextual", "—"),
    (2019, "Gonen & Goldberg", "debias esconde, não remove", "análise/ética", "09"),
], columns=["ano", "trabalho", "ideia", "família", "notebook"])
timeline

,ano,trabalho,ideia,família,notebook
0,1954,Harris; Firth (1957),hipótese distribucional,premissa,01
1,1990,Deerwester et al. (LSA),SVD da matriz termo×documento,contagem,01
2,1996,Lund & Burgess (HAL),matriz palavra×palavra por janela,contagem,01
3,2003,Bengio et al. (NNLM),modelo neural de linguagem com embeddings,preditivo,02
4,2013,Mikolov et al. (2013a),CBOW e Skip-gram; analogias,preditivo,02
5,2013,Mikolov et al. (2013b),"NEG, HS, subamostragem, frases, composição aditiva",preditivo,03
6,2014,Pennington et al. (GloVe),mínimos quadrados ponderados sobre log X_ij,contagem+predição,04
7,2014,Le & Mikolov (Doc2Vec),vetor por parágrafo,preditivo,11
8,2014,Levy & Goldberg,SGNS = fatoração de PMI deslocado,teoria,01/03
9,2016,Bolukbasi et al.,viés de gênero como direção; hard/soft debias,análise/ética,08/09/10


## 2. Quadro consolidado dos modelos deste projeto

Reunimos, para cada modelo disponível, as métricas calculadas nos notebooks anteriores (recalculadas aqui de forma rápida): analogias PT-BR (cobertura e acurácia), POS tagging, mini-STS, DirectBias de gênero (profissões epicenas + áreas). É a "tabela-resumo" para a apresentação.

In [3]:
import time
from replang.data.embeddings import load_ptwiki, load_local_model
from replang.models.trainers import list_models
from replang.data.analogies import load_analogies
from replang.eval.analogies import evaluate_analogies
from replang.eval import pos_tagging_eval, sentence_similarity_eval, MINI_STS_PT
from replang.data.corpora import load_macmorpho
from replang.data.lexicons import PT
from replang.bias import gender_direction, direct_bias
lex = PT.get(); an = load_analogies("pt-br")
train = load_macmorpho(split="train"); test = load_macmorpho(split="test", limit_sentences=300)
models = {"Wikipedia2Vec PT 100d": load_ptwiki()}
for m in list_models():
    if m["algo"] in ("word2vec", "fasttext") or m["algo"].startswith("glove"):
        models[m["name"]] = load_local_model(m["name"])
from replang.eval.parallel import parallel_map
def _row(item):
    name, wv = item
    t = time.time()
    r = evaluate_analogies(wv, an, restrict=30000, max_per_category=200).set_index("categoria")
    pos = pos_tagging_eval(wv, train, test, max_train_tokens=20000)["acurácia"]
    sts = sentence_similarity_eval(wv, MINI_STS_PT)["pearson"]
    g = gender_direction(wv, lex.definitional_pairs)
    neutras = [w for w in lex.professions + lex.stereotype_neutral if w in wv]
    return {"modelo": name, "dim": wv.dim, "|V|": len(wv), "cobertura analogias": r.loc["TOTAL", "cobertas"] / r.loc["TOTAL", "n"], "analogias (acurácia)": r.loc["TOTAL", "acurácia"],
            "POS": pos, "mini-STS ρ": sts, "DirectBias gênero": direct_bias(wv, neutras, g), "s": round(time.time() - t)}
rows = parallel_map(_row, list(models.items()))
resumo = pd.DataFrame(rows).set_index("modelo")
resumo.style.format({"cobertura analogias": "{:.0%}", "analogias (acurácia)": "{:.1%}", "POS": "{:.3f}", "mini-STS ρ": "{:.2f}", "DirectBias gênero": "{:.3f}"}).background_gradient(cmap="Blues", subset=["analogias (acurácia)", "POS", "mini-STS ρ"]).background_gradient(cmap="Reds", subset=["DirectBias gênero"])

,dim,|V|,cobertura analogias,analogias (acurácia),POS,mini-STS ρ,DirectBias gênero,s
modelo,,,,,,,,
Wikipedia2Vec PT 100d,100,50000,39%,56.8%,0.878,0.65,0.113,3
legal_cbow100,100,22203,7%,2.5%,0.843,0.24,0.147,3
legal_ft100,100,22203,7%,33.4%,0.840,0.30,0.119,3
legal_sg100,100,22203,7%,1.8%,0.832,0.35,0.119,3
machado_cbow100,100,23194,21%,2.6%,0.815,0.18,0.141,3
machado_ft100,100,23194,21%,40.7%,0.849,0.07,0.169,2
machado_glove100,100,23194,21%,2.4%,0.796,0.26,0.387,2
machado_sg100,100,23194,21%,5.3%,0.841,0.42,0.123,2
machado_sg100_hs,100,23194,21%,4.2%,0.801,0.46,0.134,2


## 3. As três perguntas do título

### Como transformar palavra em vetor?
Contando contextos (01, 04) ou prevendo-os (02, 03, 05) — ambos são formas de aplicar a hipótese distribucional; o resultado é um espaço onde **proximidade = contexto parecido** e **direções = relações** (analogias, gênero, tempo verbal).

### O que esse vetor carrega?
* Semântica lexical e relações (02, 03, 04): *rei–rainha*, *país–capital*.
* Morfologia, se o modelo olha dentro da palavra (05).
* Tópico/estilo, se olha o documento (11).
* Sintaxe e sentido dependentes do contexto, se olha a sentença (12).
* E as **estatísticas sociais do corpus** (08, 10): estereótipos ocupacionais, viés indireto — "o vetor carrega o que o texto carrega".

### O viés que vem junto: dá para tirar?
Em parte (09): Neutralize/Equalize zeram métricas específicas e preservam a utilidade; mas a estrutura residual permanece (Gonen & Goldberg) e a definição do que é "neutro" é uma decisão humana, dependente da língua (10) e da aplicação.

## 4. Roteiro de 4 horas (resumo; detalhe em `docs/roteiro_4h.md`)

| Hora | Bloco | Material | Dinâmica |
|---|---|---|---|
| 0:00 | Abertura: o título como três perguntas | nb 00 | demonstração `homem:rei::mulher:?` e `ele:médico::ela:?` no app (página 4) |
| 0:10 | A. Premissas | nb 01; app p. 1 | construir a matriz de coocorrência ao vivo com o toy corpus; PPMI; SVD |
| 0:35 | B. word2vec | nb 02, 03; app p. 2, 3, 5 | treinar Skip-gram ao vivo; HS × NEG; subamostragem; frases; PCA país→capital |
| 1:20 | C. Variações | nb 04, 05, 11; app p. 6 | GloVe (razões de probabilidade); fastText (OOV ao vivo); Doc2Vec (gêneros de Machado) |
| 1:50 | *intervalo* | | |
| 2:05 | D. Português | nb 06, 07; app p. 7 | corpus e pré-processamento de Hartmann; analogias × POS × STS; discussão de Faruqui |
| 2:45 | E. Viés | nb 08, 09, 10; app p. 8 | direção de gênero (PCA); extremos; analogias geradas; hard/soft debias; gênero gramatical em PT; experimento sintético |
| 3:35 | F. Contexto | nb 12; app p. 9 | *banco* em duas sentenças; camadas × tarefas |
| 3:50 | Síntese e discussão | nb 13 | tabela consolidada; perguntas abertas |

### Perguntas para fechar a discussão
1. Se um embedding "só reflete o corpus", de quem é a responsabilidade pelo viés? (dados, modelo, aplicação)
2. O que uma avaliação de embeddings **para o português** deveria medir que os benchmarks traduzidos não medem?
3. Com modelos contextuais e LLMs, o que muda nessa história — e o que permanece?

## Referências completas da disciplina

### Leituras obrigatórias
- MIKOLOV, T.; CHEN, K.; CORRADO, G.; DEAN, J. **Efficient Estimation of Word Representations in Vector Space.** ICLR Workshop, 2013. [arXiv:1301.3781](https://arxiv.org/abs/1301.3781)
- BOLUKBASI, T.; CHANG, K.-W.; ZOU, J.; SALIGRAMA, V.; KALAI, A. **Man is to Computer Programmer as Woman is to Homemaker? Debiasing Word Embeddings.** NIPS, 2016. [arXiv:1607.06520](https://arxiv.org/abs/1607.06520)
- HARTMANN, N. et al. **Portuguese Word Embeddings: Evaluating on Word Analogies and Natural Language Tasks.** STIL, 2017. [arXiv:1708.06025](https://arxiv.org/abs/1708.06025)

### Leituras complementares
- MIKOLOV, T.; SUTSKEVER, I.; CHEN, K.; CORRADO, G.; DEAN, J. **Distributed Representations of Words and Phrases and their Compositionality.** NIPS, 2013. [arXiv:1310.4546](https://arxiv.org/abs/1310.4546)
- PENNINGTON, J.; SOCHER, R.; MANNING, C. **GloVe: Global Vectors for Word Representation.** EMNLP, 2014. [ACL D14-1162](https://aclanthology.org/D14-1162/)
- BOJANOWSKI, P.; GRAVE, E.; JOULIN, A.; MIKOLOV, T. **Enriching Word Vectors with Subword Information.** TACL, 2017. [arXiv:1607.04606](https://arxiv.org/abs/1607.04606)
- PETERS, M. et al. **Deep contextualized word representations.** NAACL, 2018. [arXiv:1802.05365](https://arxiv.org/abs/1802.05365)
- LE, Q.; MIKOLOV, T. **Distributed Representations of Sentences and Documents.** ICML, 2014. [arXiv:1405.4053](https://arxiv.org/abs/1405.4053)

### Citadas ao longo dos notebooks
- HARRIS, Z. **Distributional Structure.** Word, 1954; FIRTH, J. R. *A synopsis of linguistic theory*, 1957.
- LEVY, O.; GOLDBERG, Y. **Neural Word Embedding as Implicit Matrix Factorization.** NIPS, 2014; LEVY, GOLDBERG & DAGAN. **Improving Distributional Similarity with Lessons Learned from Word Embeddings.** TACL, 2015.
- BARONI, M.; DINU, G.; KRUSZEWSKI, G. **Don't count, predict!** ACL, 2014.
- FARUQUI, M. et al. **Problems With Evaluation of Word Embeddings Using Word Similarity Tasks.** RepEval, 2016.
- RODRIGUES, J. et al. **LX-DSemVectors: Distributional Semantics Models for Portuguese.** PROPOR, 2016.
- GONEN, H.; GOLDBERG, Y. **Lipstick on a Pig.** NAACL, 2019.
- LING, W. et al. **Two/Too Simple Adaptations of Word2Vec for Syntax Problems** (wang2vec). NAACL, 2015.
- DEVLIN, J. et al. **BERT: Pre-training of Deep Bidirectional Transformers for Language Understanding.** NAACL, 2019.